In [1]:
import numpy as np
import pandas as pd

============================================================
TRANSFORMAÇÃO AFIM: PIXEL -> UTM
7 pontos para ajuste e 3 pontos para teste
============================================================

In [2]:
# ------------------------------------------------------------
# 1. DADOS
# ------------------------------------------------------------
pontos = [1, 2, 5, 6, 8, 9, 10, 11, 12, 13]

In [3]:
x_pixel = np.array([1208, 1157,  134,  846, 2238,  518, 2418, 2639, 1524, 2678], dtype=float)
y_pixel = np.array([2414, 1735,  353, 1073, 1317, 1641,  244, 2598, 1019, 1050], dtype=float)

In [4]:
E_utm = np.array([
    681100.209,
    681079.091,
    680377.816,
    680888.011,
    681875.866,
    680626.292,
    682031.497,
    682134.395,
    681361.203,
    682208.231
], dtype=float)

In [5]:
N_utm = np.array([
    7464305.984,
    7464791.902,
    7465806.071,
    7465251.245,
    7465066.854,
    7464876.497,
    7465841.798,
    7464127.842,
    7465299.251,
    7465250.939
], dtype=float)

In [6]:
# ------------------------------------------------------------
# 2. SEPARAÇÃO: 7 PARA AJUSTE, 3 PARA TESTE
# ------------------------------------------------------------
idx_ajuste = np.array([0, 1, 2, 3, 4, 5, 6])   # P1, P2, P5, P6, P8, P9, P10
idx_teste  = np.array([7, 8, 9])               # P11, P12, P13

In [7]:
x_aj = x_pixel[idx_ajuste]
y_aj = y_pixel[idx_ajuste]
E_aj = E_utm[idx_ajuste]
N_aj = N_utm[idx_ajuste]

In [8]:
x_te = x_pixel[idx_teste]
y_te = y_pixel[idx_teste]
E_te = E_utm[idx_teste]
N_te = N_utm[idx_teste]

In [9]:
pontos_aj = np.array(pontos)[idx_ajuste]
pontos_te = np.array(pontos)[idx_teste]

------------------------------------------------------------
3. AJUSTE DA TRANSFORMAÇÃO AFIM
------------------------------------------------------------
Modelo:
E = a0 + a1*x + a2*y
N = b0 + b1*x + b2*y

In [10]:
A = np.column_stack((np.ones(len(x_aj)), x_aj, y_aj))

In [11]:
param_E = np.linalg.inv(A.T @ A) @ (A.T @ E_aj)
param_N = np.linalg.inv(A.T @ A) @ (A.T @ N_aj)

In [12]:
a0, a1, a2 = param_E
b0, b1, b2 = param_N

In [13]:
print("=== TRANSFORMAÇÃO AFIM (PIXEL -> UTM) ===")
print(f"E = {a0:.6f} + {a1:.6f}*x + {a2:.6f}*y")
print(f"N = {b0:.6f} + {b1:.6f}*x + {b2:.6f}*y")
print()

=== TRANSFORMAÇÃO AFIM (PIXEL -> UTM) ===
E = 680295.489184 + 0.721354*x + -0.026758*y
N = 7466056.175287 + -0.018630*x + -0.717120*y



In [14]:
# ------------------------------------------------------------
# 4. RESÍDUOS NOS PONTOS DE AJUSTE
# ------------------------------------------------------------
E_calc_aj = a0 + a1 * x_aj + a2 * y_aj
N_calc_aj = b0 + b1 * x_aj + b2 * y_aj

In [15]:
vE_aj = E_aj - E_calc_aj
vN_aj = N_aj - N_calc_aj

In [16]:
RMS_E_aj = np.sqrt(np.mean(vE_aj**2))
RMS_N_aj = np.sqrt(np.mean(vN_aj**2))
RMS_plani_aj = np.sqrt(np.mean(vE_aj**2 + vN_aj**2))

In [17]:
print("=== AJUSTE: 7 PONTOS ===")
print(f"RMS E: {RMS_E_aj:.4f} m")
print(f"RMS N: {RMS_N_aj:.4f} m")
print(f"RMS planimétrico: {RMS_plani_aj:.4f} m")
print()

=== AJUSTE: 7 PONTOS ===
RMS E: 4.9999 m
RMS N: 8.6234 m
RMS planimétrico: 9.9681 m



In [18]:
df_aj = pd.DataFrame({
    "Ponto": pontos_aj,
    "x_pixel": x_aj,
    "y_pixel": y_aj,
    "E_obs": E_aj,
    "N_obs": N_aj,
    "E_calc": E_calc_aj,
    "N_calc": N_calc_aj,
    "vE": vE_aj,
    "vN": vN_aj
})

In [19]:
print("=== RESÍDUOS DOS PONTOS DE AJUSTE ===")
print(df_aj.to_string(index=False))
print()

=== RESÍDUOS DOS PONTOS DE AJUSTE ===
 Ponto  x_pixel  y_pixel      E_obs       N_obs        E_calc       N_calc        vE         vN
     1   1208.0   2414.0 681100.209 7464305.984 681102.290767 7.464303e+06 -2.081767   3.442569
     2   1157.0   1735.0 681079.091 7464791.902 681083.670379 7.464790e+06 -4.579379   1.485686
     5    134.0    353.0 680377.816 7465806.071 680382.705024 7.465801e+06 -4.889024   5.535653
     6    846.0   1073.0 680888.011 7465251.245 680877.043140 7.465271e+06 10.967860 -19.698992
     8   2238.0   1317.0 681875.866 7465066.854 681874.638566 7.465070e+06  1.227434  -3.179432
     9    518.0   1641.0 680626.292 7464876.497 680625.240604 7.464870e+06  1.051396   6.766696
    10   2418.0    244.0 682031.497 7465841.798 682033.193519 7.465836e+06 -1.696519   5.647819



In [20]:
# ------------------------------------------------------------
# 5. TESTE NOS 3 PONTOS
# ------------------------------------------------------------
E_calc_te = a0 + a1 * x_te + a2 * y_te
N_calc_te = b0 + b1 * x_te + b2 * y_te

In [21]:
vE_te = E_te - E_calc_te
vN_te = N_te - N_calc_te

In [22]:
RMS_E_te = np.sqrt(np.mean(vE_te**2))
RMS_N_te = np.sqrt(np.mean(vN_te**2))
RMS_plani_te = np.sqrt(np.mean(vE_te**2 + vN_te**2))

In [23]:
print("=== TESTE: 3 PONTOS ===")
print(f"RMS E: {RMS_E_te:.4f} m")
print(f"RMS N: {RMS_N_te:.4f} m")
print(f"RMS planimétrico: {RMS_plani_te:.4f} m")
print()

=== TESTE: 3 PONTOS ===
RMS E: 6.9568 m
RMS N: 9.4760 m
RMS planimétrico: 11.7555 m



In [24]:
df_te = pd.DataFrame({
    "Ponto": pontos_te,
    "x_pixel": x_te,
    "y_pixel": y_te,
    "E_obs": E_te,
    "N_obs": N_te,
    "E_calc": E_calc_te,
    "N_calc": N_calc_te,
    "vE": vE_te,
    "vN": vN_te
})

In [25]:
print("=== RESULTADOS DOS PONTOS DE TESTE ===")
print(df_te.to_string(index=False))

=== RESULTADOS DOS PONTOS DE TESTE ===
 Ponto  x_pixel  y_pixel      E_obs       N_obs        E_calc       N_calc        vE         vN
    11   2639.0   2598.0 682134.395 7464127.842 682129.624466 7.464144e+06  4.770534 -16.089518
    12   1524.0   1019.0 681361.203 7465299.251 681367.565886 7.465297e+06 -6.362886   2.213757
    13   2678.0   1050.0 682208.231 7465250.939 682199.178573 7.465253e+06  9.052427  -2.368305
